# Agente inversor: protocolo de entrenamiento y evaluación

1. **Separación temporal** de los datos
2. **Qué datos se usaron para cada decisión** del proyecto
3. **Controles automáticos** contra el uso de información del futuro
4. **El agente final, congelado** antes de tocar el período held-out
5. **Plan de la evaluación final**, fijado antes de ver sus resultados

In [1]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

# Ejecutar desde la raíz del repositorio, como la CLI.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import gymnasium as gym
import pandas as pd

from rl_games_sb import envs, investor  # noqa: F401  (envs registra los entornos)

FINAL = json.loads(Path("configs/investor_final.json").read_text(encoding="utf-8"))

## 1. Separación temporal

```
Datos GOOGL diarios, 2009-05-22 → 2018-08-29
│
├── Desarrollo
│   ├── Entrenamiento   2009-05-22 → 2014-12-31   el agente aprende; se ajusta la normalización
│   └── Validación      2015-01-01 → 2015-12-31   se comparan recompensas y se elige κ
│
└── Held-out (test)     2016-01-01 → 2018-08-29   evaluación final, una sola vez
```

El esquema de la actividad tiene dos bloques, entrenamiento y held-out. Aquí
el bloque de entrenamiento se divide en dos: los años 2009–2014 para aprender
y 2015 para comparar variantes. Así las decisiones de diseño se toman con
datos que el agente no vio al entrenar, **sin gastar el held-out**, que se
reserva completo para la evaluación final.

In [2]:
df = investor.load()
rows = {}
for p in investor.SPLITS:
    rows[p] = {
        "desde": investor.split(df, p).index[0].date(),
        "hasta": investor.split(df, p).index[-1].date(),
        "días con datos": len(investor.split(df, p)),
        "decisiones del agente": len(investor.InvestorEnv(period=p).dates) - 1,
    }
pd.DataFrame(rows).T

,desde,hasta,días con datos,decisiones del agente
train,2009-05-22,2014-12-31,1413,1363
valid,2015-01-02,2015-12-31,252,251
test,2016-01-04,2018-08-29,670,669


Train tiene 1.413 días con datos, pero el agente toma 1.363 decisiones, porque
los indicadores necesitan 49 días previos para calcularse. Validación y test
toman esos días previos del período anterior, que ya era pasado en ese
momento, así que no pierden días.

## 2. Qué datos informaron cada decisión

| Decisión | Parte | Datos usados para tomarla |
|---|---|---|
| Activo y frecuencia (GOOGL diario) | 6 | Ninguno: es el dataset común del curso |
| Fechas de la división | 6 | Ninguno: años completos, fijados antes de entrenar |
| Indicadores del estado | 4 y 7 | Elegidos a priori; su capacidad de predicción se midió solo en train |
| Normalización de los indicadores | 4 | Media y desviación de train |
| Espacio de acciones {0 %, 50 %, 100 %} | 5 | Recomendación de la actividad |
| Cambiar de la recompensa v0 a v1 | 9 | Comportamiento en train y validación |
| Penalización κ = 400 bps | 9 | Validación (mejor crecimiento medio de 3 semillas) |
| Hiperparámetros y arquitectura de DQN | 10 | Fijados a priori, sin ajuste |
| Criterio de parada | 10 | Presupuesto fijo de 300.000 pasos |
| Semillas 0, 1 y 2 | 10 | Fijadas antes de entrenar |
| Agente final: DQN v1, 3 semillas | 12 | Validación |

**Ninguna decisión usó el período de test.**

**Lo que sí se vio del test, para transparencia:**

- **Análisis exploratorio (parte 7).** Describe la serie completa: el precio,
  la volatilidad y los drawdowns de 2016–2018, el retorno de comprar y
  mantener en test (+66,5 %) y la distribución de los indicadores normalizados
  en test. Son estadísticas del activo, no de ningún agente, y ninguna
  decisión del diseño cambió por ellas.
- **Pruebas del código.** Algunas pruebas usan el período de test con
  políticas fijas (siempre 0 %, siempre 100 %) para verificar la contabilidad
  del entorno, y el chequeo inicial del entorno (parte 8) corrió en test
  políticas fijas y un agente aleatorio. Ninguna evaluó al agente DQN.

El agente DQN nunca se ha ejecutado sobre el período de test.

## 3. Controles automáticos contra fugas de información

| Prueba | Qué garantiza |
|---|---|
| `test_future_prices_do_not_change_past_features` | Los indicadores del día *t* no cambian si se alteran los precios posteriores a *t* |
| `test_scaler_ignores_the_evaluation_period` | La normalización es la misma aunque cambien los precios de validación y test |
| `test_env_does_not_see_future_prices` | Las observaciones y recompensas del entorno hasta el día *k* no cambian si se alteran los precios posteriores; el paso siguiente sí, lo que demuestra que la prueba detectaría una fuga |
| `test_periods_are_disjoint_ordered_and_complete` | Train, validación y test no se solapan y van en orden |
| `test_episode_covers_the_period_and_truncates` | Cada episodio termina en el último día de su período, sin usar el primer precio del siguiente |

In [3]:
# check=True: si alguna prueba falla, esta celda falla y el notebook se detiene.
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"],
                        capture_output=True, text=True, check=True)
print(result.stdout.strip().splitlines()[-1])

20 passed in 2.11s


## 4. El agente final, congelado

El agente final es **DQN con la recompensa v1** (log-retorno neto de costos
menos 400 bps por unidad de posición cambiada), entrenado solo en train, con
las **3 semillas**. No se elige una semilla: se evalúan las tres.

Sus modelos y su configuración quedaron registrados en
`configs/investor_final.json`, con el hash SHA-256 de cada archivo, **antes**
de usar el período de test. La celda siguiente comprueba que los archivos
que se van a evaluar son exactamente esos: si alguien reentrenara o
modificara un modelo después de ver el test, su hash cambiaría.

In [4]:
def sha256(path, text=False):
    data = Path(path).read_bytes()
    if text:  # el hash del texto no depende de los saltos de línea (CRLF en Windows)
        data = data.replace(b"\r\n", b"\n")
    return hashlib.sha256(data).hexdigest()


checks = {"configuración": sha256(FINAL["config"]["path"], text=True) == FINAL["config"]["sha256_lf"]}
for seed, entry in FINAL["models"].items():
    checks[f"modelo semilla {seed}"] = sha256(entry["path"]) == entry["sha256"]
checks["penalización del entorno"] = gym.spec(FINAL["env"]).kwargs == FINAL["env_kwargs"]
checks["costo de transacción"] = investor.COST_BPS == FINAL["cost_bps"]
checks["fechas de los períodos"] = {k: list(v) for k, v in investor.SPLITS.items()} == FINAL["splits"]

print(f"Congelado en el commit {FINAL['frozen_at_commit']}")
for name, ok in checks.items():
    print(f"  {'OK   ' if ok else 'FALLA'} {name}")
assert all(checks.values()), "El agente no coincide con el que se congeló: no evaluar en test"

Congelado en el commit aade48c
  OK    configuración
  OK    modelo semilla 0
  OK    modelo semilla 1
  OK    modelo semilla 2
  OK    penalización del entorno
  OK    costo de transacción
  OK    fechas de los períodos


**Por qué no se reentrena con train + validación.** Una práctica común es,
después de elegir los hiperparámetros, reentrenar con todos los datos de
desarrollo. Aquí no se hace: el modelo reentrenado sería distinto del que se
validó, nadie habría observado su comportamiento, y la normalización y κ se
fijaron con train como base. Se evalúan exactamente los modelos cuyo
comportamiento se analizó en las partes 9 a 11.

## 5. Plan de la evaluación final (fijado antes de ver resultados)

| | |
|---|---|
| **Período** | Test: 2016-01-04 → 2018-08-29, 669 decisiones |
| **Agente** | Los 3 modelos de `configs/investor_final.json`, política greedy (sin exploración) |
| **Baselines** | Comprar y mantener; aleatorio con las semillas 0 a 199; siempre en efectivo; siempre 50 % |
| **Condiciones** | Las mismas para todos: $100.000 en efectivo al inicio, 10 bps por operación, ejecución al cierre |
| **Métricas** | Retorno total y anual, volatilidad, Sharpe, drawdown máximo, operaciones, costos y exposición media |
| **Cómo se reporta** | Cada semilla, su media y la peor semilla; sin elegir la mejor |
| **Pregunta** | ¿El agente supera a comprar y mantener en retorno, en riesgo (drawdown, Sharpe) y en costos? ¿En qué percentil queda frente a los 200 agentes aleatorios? |

**Compromisos:**

- La evaluación en test se ejecuta **una sola vez**, en un notebook aparte.
- Después de verla **no se cambia nada** del agente: ni la recompensa, ni κ,
  ni los hiperparámetros, ni la elección de semillas.
- El resultado se reporta tal como salga. La actividad lo dice: un resultado
  negativo no implica un mal proyecto.
- Si alguna parte posterior de la actividad pide otra métrica, se calcula
  sobre estas mismas ejecuciones, sin volver a decidir nada.